# Xử lý trắc nghiệm

In [2]:
import pandas as pd 

## Q1. 
Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [16]:
df = pd.read_csv("../data/raw/orders.csv", encoding="utf-8") 

df = df[df["order_status"] == "delivered"].copy().drop_duplicates(subset=["customer_id", "order_date"])

df["order_date"] = pd.to_datetime(df["order_date"])

df = df.sort_values(by=["customer_id", "order_date"])

df["order_gap"] = df.groupby("customer_id")["order_date"].diff().dt.days

median_gap = df["order_gap"].dropna().median()

print(median_gap) # 178 -> C 

df.head(10)

178.0
Empty DataFrame
Columns: [order_id, order_date, customer_id, zip, order_status, payment_method, device_type, order_source, order_gap]
Index: []


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,order_gap
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search,NaN
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,1101.0
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,632.0
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,1037.0
91192,117673,2013-09-20,2,15201,delivered,paypal,mobile,social_media,NaN
632148,815219,2022-07-06,2,15201,delivered,apple_pay,mobile,paid_search,3211.0
10063,13047,2012-08-27,3,15201,delivered,credit_card,mobile,social_media,NaN
24017,31033,2012-11-24,3,15201,delivered,apple_pay,desktop,social_media,89.0
560485,722725,2020-06-28,4,15201,delivered,credit_card,mobile,social_media,NaN
7135,9294,2012-08-09,5,15201,delivered,credit_card,desktop,paid_search,NaN


## Q2. 
Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [21]:
df = pd.read_csv("../data/raw/products.csv") 

df["gross_margin"] = (df["price"] - df["cogs"]) / df["price"] 

segment_margin = df.groupby("segment")["gross_margin"].mean().sort_values(ascending=False) 

print(segment_margin)

print(segment_margin.idxmax())
df.head(5)

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64
Standard


,product_id,product_name,category,segment,size,color,price,cogs,gross_margin
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278,0.2871
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954,0.4558
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406,0.1080


## Q3. 
Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [4]:
df_ret = pd.read_csv("../data/raw/returns.csv")
df_prod = pd.read_csv("../data/raw/products.csv")

merged = pd.merge(df_ret, df_prod[["product_id", "category"]], on="product_id", how="inner")

streetwear_returns = merged[merged["category"] == "Streetwear"]

return_reason_counts = streetwear_returns.groupby("return_reason").size().sort_values(ascending=False)
print(return_reason_counts)

most_common_reason = return_reason_counts.idxmax()
count_most_common = return_reason_counts.max()
print(f"\nLý do trả hàng xuất hiện nhiều nhất: \"{most_common_reason}\" với {count_most_common} lần")


return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
dtype: int64

Lý do trả hàng xuất hiện nhiều nhất: "wrong_size" với 7626 lần


## Q4. 
Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [6]:
pd_web = pd.read_csv("../data/raw/web_traffic.csv") 

traffic_stats = pd_web.groupby('traffic_source')['bounce_rate'].mean().sort_values()

print(traffic_stats)

best_source = traffic_stats.idxmin()
print(f"{best_source}")

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
email_campaign


## Q5. 
Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [ ]:
pd_ot = pd.read_csv("../data/raw/order_items.csv") 

promo_count = pd_ot["promo_id"].notnull().sum()

total_rows = len(pd_ot)

percentage = (promo_count / total_rows) * 100

print(f"Số dòng có khuyến mãi: {promo_count}")
print(f"Tổng số dòng: {total_rows}")
print(f"Tỷ lệ phần trăm xấp xỉ: {percentage:.2f}%")

Số dòng có khuyến mãi: 276316
Tổng số dòng: 714669
Tỷ lệ phần trăm xấp xỉ: 38.66%


C:\Users\ms24\AppData\Local\Temp\ipykernel_24472\361263545.py:1: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_ot = pd.read_csv("../data/raw/order_items.csv")


## Q6. 
Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [9]:
# 1. Đọc dữ liệu
df_customers = pd.read_csv("../data/raw/customers.csv")
df_orders = pd.read_csv("../data/raw/orders.csv")

# 2. Lọc khách hàng có age_group khác null
df_customers = df_customers[df_customers["age_group"].notnull()]

# 3. Đếm số đơn hàng của mỗi khách hàng từ bảng orders
order_counts = df_orders.groupby("customer_id").size().reset_index(name="total_orders")

# 4. Join thông tin số đơn hàng vào bảng khách hàng
df_merged = pd.merge(df_customers, order_counts, on="customer_id", how="left")

# Những khách hàng không có đơn nào trong orders.csv sẽ bị NaN, ta điền là 0
df_merged["total_orders"] = df_merged["total_orders"].fillna(0)

# 5. Tính toán theo từng nhóm tuổi (age_group)
# Tổng số đơn / Số khách hàng trong nhóm
age_group_stats = df_merged.groupby("age_group").agg(
    total_orders_in_group = ("total_orders", "sum"),
    customer_count = ("customer_id", "count")
)

age_group_stats["avg_orders_per_customer"] = age_group_stats["total_orders_in_group"] / age_group_stats["customer_count"]

# 6. Sắp xếp để tìm nhóm cao nhất
result = age_group_stats.sort_values(by="avg_orders_per_customer", ascending=False)

print(result)
print(f"\n=> Nhóm tuổi có số đơn hàng trung bình cao nhất là: {result.index[0]}")

           total_orders_in_group  customer_count  avg_orders_per_customer
age_group                                                                
55+                      72760.0           13457                 5.406851
45-54                   124138.0           23172                 5.357241
35-44                   170368.0           31920                 5.337343
25-34                   190622.0           36342                 5.245226
18-24                    89057.0           17039                 5.226656

=> Nhóm tuổi có số đơn hàng trung bình cao nhất là: 55+


## Q7. 
Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales.csv?

In [13]:
df_geo = pd.read_csv("../data/raw/geography.csv")
df_ord = pd.read_csv("../data/raw/orders.csv")
df_oit = pd.read_csv("../data/raw/order_items.csv")


# 1. Tính doanh thu trên mỗi dòng của order_items
df_oit['revenue'] = df_oit['quantity'] * df_oit['unit_price']

# 2. Join để lấy mã zip từ bảng orders
df_step1 = pd.merge(df_oit, df_ord[['order_id', 'zip']], on='order_id', how='inner')
# print(df_step1)
# 3. Join để lấy region từ bảng geography
df_step2 = pd.merge(df_step1, df_geo[['zip', 'region']], on='zip', how='inner')

# 4. Tính tổng doanh thu theo vùng
region_revenue = df_step2.groupby('region')['revenue'].sum().sort_values(ascending=False)

print(region_revenue)
print(f"\n=> Vùng tạo ra doanh thu cao nhất là: {region_revenue.idxmax()}")

region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: revenue, dtype: float64

=> Vùng tạo ra doanh thu cao nhất là: East


C:\Users\ms24\AppData\Local\Temp\ipykernel_7148\1035578194.py:3: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_oit = pd.read_csv("../data/raw/order_items.csv")


## Q8. 
Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [ ]:
# 1. Đọc dữ liệu
df_orders = pd.read_csv("../data/raw/orders.csv")

# 2. Lọc các đơn hàng có trạng thái là "cancelled"
df_cancelled = df_orders[df_orders["order_status"] == "cancelled"]

# 3. Đếm số lần xuất hiện của từng phương thức thanh toán (payment_method)
payment_counts = df_cancelled["payment_method"].value_counts()

# 4. In kết quả
print("Thống kê phương thức thanh toán của các đơn bị hủy:")
print(payment_counts)

# Lấy phương thức xuất hiện nhiều nhất
top_payment = payment_counts.idxmax()
print(f"\n=> Phương thức thanh toán được sử dụng nhiều nhất trong các đơn bị hủy là: {top_payment}")

Thống kê phương thức thanh toán của các đơn bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

=> Phương thức thanh toán được sử dụng nhiều nhất trong các đơn bị hủy là: credit_card


## Q9. 
Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

In [ ]:
# 1. Đọc dữ liệu
df_ret = pd.read_csv("../data/raw/returns.csv")
df_ot = pd.read_csv("../data/raw/order_items.csv")
df_prod = pd.read_csv("../data/raw/products.csv")

# 2. Join bảng returns với products để biết kích cỡ của từng món đồ bị trả
df_ret_prod = pd.merge(df_ret, df_prod[["product_id", "size"]], on="product_id", how="left")
# Đếm số lượng trả hàng theo từng size
returns_by_size = df_ret_prod.groupby("size").size()

# 3. Join bảng order_items với products để biết kích cỡ của từng món đồ được bán
df_ot_prod = pd.merge(df_ot, df_prod[["product_id", "size"]], on="product_id", how="left")
# Đếm số lượng dòng order theo từng size
orders_by_size = df_ot_prod.groupby("size").size()

# 4. Tính tỷ lệ trả hàng
# Chuyển về DataFrame để dễ quan sát
stats = pd.DataFrame({
    "returns_count": returns_by_size,
    "orders_count": orders_by_size
})

stats["return_rate"] = stats["returns_count"] / stats["orders_count"]

# 5. Sắp xếp để tìm kích cỡ có tỷ lệ cao nhất
result = stats.sort_values(by="return_rate", ascending=False)

print(result)
print(f"\n=> Kích thước có tỷ lệ trả hàng cao nhất là: {result.index[0]}")

      returns_count  orders_count  return_rate
size                                          
S              9723        172042     0.056515
L              9741        173174     0.056250
M              9820        176428     0.055660
XL            10655        193025     0.055200

=> Kích thước có tỷ lệ trả hàng cao nhất là: S


C:\Users\ms24\AppData\Local\Temp\ipykernel_7148\2894345314.py:5: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ot = pd.read_csv("../data/raw/order_items.csv")


## Q10. 
Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

In [4]:
# 1. Đọc dữ liệu
df_pay = pd.read_csv("../data/raw/payments.csv")

# 2. Gom nhóm theo số kỳ trả góp (installments) và tính trung bình giá trị thanh toán (payment_value)
# Dùng tên cột chính xác từ ảnh: 'installments' và 'payment_value'
avg_payment_per_plan = df_pay.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

# 3. Hiển thị kết quả
print("Giá trị thanh toán trung bình theo kế hoạch trả góp:")
print(avg_payment_per_plan)

# 4. Lấy kế hoạch có giá trị cao nhất
top_plan = avg_payment_per_plan.idxmax()
print(f"\n=> Kế hoạch trả góp có giá trị trung bình cao nhất là: {top_plan} kỳ")

Giá trị thanh toán trung bình theo kế hoạch trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

=> Kế hoạch trả góp có giá trị trung bình cao nhất là: 6 kỳ
